In [1]:
import geopandas as gpd
import pandas as pd
import os

In [5]:
# --- 1. CONFIGURACIÓN DE RUTAS ---
# Rutas de los archivos resultantes de los pasos anteriores
ruta_mov_activa = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1V1_Movilidad_activa\1_D4C1V1_output_Movilidad_activa.gpkg"
ruta_transporte = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1V2_output_Transporte_público.gpkg"
ruta_vialidad = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1V3_Estructura_vial\1_D4C1V3_output_Estructura_vial.gpkg"

carpeta_salida = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\2_D4C1_Movilidad"
nombre_archivo_final = "2_D4C1_Movilidad_output.gpkg"



In [17]:
# --- 1. CONFIGURACIÓN DE RUTAS CORREGIDAS ---
# Nota: Verifica que estas carpetas realmente existan en tu Escritorio
ruta_mov_activa = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1V1_Movilidad_activa\1_D4C1V1_output_Movilidad_activa.gpkg"

# Corregida: Quitamos la subcarpeta extra si es que lo guardaste directo en outputs
ruta_transporte = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1V2_Transporte_público\1_D4C1V2_output_Transporte_público.gpkg"

# Corregida: Ruta de Vialidad
ruta_vialidad = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1V3_Estructura_vial\1_D4C1V3_output_Estructura_vial.gpkg"

carpeta_salida = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\2_D4C1_Movilidad"
nombre_archivo_final = "2_D4C1_Movilidad_output.gpkg"

# 2. CARGA
print("Cargando capas...")
gdf1 = gpd.read_file(ruta_mov_activa)
gdf2 = gpd.read_file(ruta_transporte)
gdf3 = gpd.read_file(ruta_vialidad)

# 3. UNIFICAR COLUMNA CATEGORÍA (Corregido el error de sintaxis 'en' -> 'in')
def rescatar_categoria(gdf):
    for col in gdf.columns:
        # Buscamos variaciones del nombre
        if col.lower() in ['categoría', 'categoria']:
            return gdf.rename(columns={col: 'CATEGORIA_FINAL'})
    return gdf

gdf1 = rescatar_categoria(gdf1)
gdf2 = rescatar_categoria(gdf2)
gdf3 = rescatar_categoria(gdf3)

# 4. UNIÓN (CONCAT)
print("Uniendo capas...")
gdf_master = pd.concat([gdf1, gdf2, gdf3], ignore_index=True)

# 5. LIMPIEZA DE NOMBRES PARA PYOGRIO
nuevos_nombres = {}
for i, col in enumerate(gdf_master.columns):
    if col == 'CATEGORIA_FINAL':
        nuevos_nombres[col] = 'Categoría'
    elif col == 'geometry':
        nuevos_nombres[col] = 'geometry'
    else:
        # Nombres cortos v_0, v_1... para evitar el FieldError: Nombre_de
        nuevos_nombres[col] = f"v_{i}"

gdf_master = gdf_master.rename(columns=nuevos_nombres)

# Asegurar GeoDataFrame y CRS
gdf_master = gpd.GeoDataFrame(gdf_master, geometry='geometry', crs=gdf1.crs)

# 6. GUARDADO
print("Guardando...")
ruta_final = os.path.join(carpeta_salida, nombre_archivo_final)

try:
    # Usamos el motor por defecto que sí tienes instalado
    gdf_master.to_file(ruta_final, driver="GPKG", promote_to_multi=True)
    print("-" * 30)
    print(f"¡POR FIN! Archivo maestro creado exitosamente.")
    print(f"Ruta: {ruta_final}")
except Exception as e:
    print(f"Híjole, falló algo: {e}")

Cargando capas...
Uniendo capas...
Guardando...
------------------------------
¡POR FIN! Archivo maestro creado exitosamente.
Ruta: C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\2_D4C1_Movilidad\2_D4C1_Movilidad_output.gpkg
